# Policy Management

Inspect device status and create/simulate/evaluate access policies from a notebook using `IamAdminClient` plus the `device_status_table` rich-display helper.

In [ ]:
from iam_sdk import IamClient, IamAdminClient
from iam_sdk.jupyter import device_status_table

BASE_URL = "https://localhost:5161"
TENANT_ID = "<building-tenant-id>"

async with IamClient(BASE_URL) as auth:
    login = await auth.login("admin@example.com", "Password123!")

admin = IamAdminClient(BASE_URL, login.access_token)

In [ ]:
devices = await admin.list_devices(tenant_id=TENANT_ID)
device_status_table(devices)

In [ ]:
# Simulate the impact before creating the policy.
simulation = await admin.simulate_policy(
    tenant_id=TENANT_ID,
    resource="buildings/*/sensors/*",
    action="read",
    effect="Allow",
    inheritance_scope="Descendants",
)
print(f"Would affect {simulation.affected_tenant_count} tenants, {simulation.affected_user_count} users")

In [ ]:
policy = await admin.create_policy(
    name="Allow sensor read (notebook)",
    tenant_id=TENANT_ID,
    resource="buildings/*/sensors/*",
    action="read",
    effect="Allow",
    inheritance_scope="Descendants",
)

evaluation = await admin.evaluate_policy(
    tenant_id=TENANT_ID, resource="buildings/hq/sensors/temp-001", action="read"
)
print(f"Access allowed: {evaluation.is_allowed} ({evaluation.reason})")